In [1]:
"""
Hotel Data Collection And Cleaning Pipeline
==========================================================
本程式為 Hotel 資料處理流程的初始階段，
從 Google API / Agoda API 進行原始資料採集，
並執行初步的資料清理。

"""

# ==========================================================
# IMPORTS
# ==========================================================

import aiohttp
import asyncio
import pandas as pd
import json
import re
import time
import random

from dataclasses import dataclass, asdict
from typing import Optional
from urllib.parse import urlparse
from google.colab import userdata, files


# ==========================================================
# SETTINGS
# 系統設定
# ==========================================================

class Settings:
  HOTEL_LIST = [
    "北投亞太飯店",
    "北投馥悅溫泉酒店",
    "曲水會館",
    "北投老爺酒店",
    "北投晶泉丰旅",
    "北投麗禧溫泉酒店",
    "大地酒店",
    "北投日勝生加賀屋",
    "北投春天酒店",
    "北投天玥泉溫泉會館",
  ]

  PLACES_KEY = userdata.get("PLACES_API_KEY")
  CX = userdata.get("CX_API_KEY")
  AGODA_API_KEY = userdata.get("AGODA_API_KEY")

  CHECKIN = "2026-03-03"
  CHECKOUT = "2026-03-04"

  PRICE_MARKUP = 1.13      # Agoda 價格加成(含稅)
  KKDAY_BAD_KEYWORD = "餐券"  # KKday 過濾字

  MAX_CONCURRENCY = 5      # 平行執行數量
  CSV_FILENAME = "hotels_output.csv"


# ==========================================================
# UTILS
# 工具函式
# ==========================================================

# ----------------------------------------------------------
# Constants
# ----------------------------------------------------------

# 預先編譯手機號碼正規表達式
MOBILE_PATTERN = re.compile(r"^09\d{8}$")

# 市話區碼（由長到短排序，避免 037 被 03 誤判）
AREA_CODES = sorted(
  ["02","03","037","04","049","05","06","07","08","082","0836","089"],
  key=len,
  reverse=True
)

# ----------------------------------------------------------
# Functions
# ----------------------------------------------------------


def normalize_phone(phone):
  # """
  # 統一電話格式
  # """

  if not phone:
    return ""

  # 移除空白與 -
  cleaned = phone.replace(" ", "").replace("-", "")

  # +886 -> 0
  if cleaned.startswith("+886"):
    cleaned = "0" + cleaned[4:]

  # 手機
  if MOBILE_PATTERN.match(cleaned):
    return f"{cleaned[:4]}-{cleaned[4:]}"

  # 市話
  for code in AREA_CODES:
    if cleaned.startswith(code):
      return f"{code}-{cleaned[len(code):]}"

  return phone


def normalize_address(address):
  # """
  # 清理地址（移除郵遞區號、空白、台灣文字）
  # """

  if not address:
    return ""

  addr = address.replace(" ", "")
  addr = re.sub(r"^[0-9]{3,5}", "", addr)
  addr = addr.replace("台灣", "").replace("臺灣", "")
  addr = re.sub(r"[0-9]{3,5}$", "", addr)

  return addr


def generate_slug(website):
  # """
  # 從網址產生 slug
  # """

  if not website:
    return ""

  try:
    parsed = urlparse(website)

    # 取得 hostname（例如：www.hotel.example.com）
    hostname = parsed.hostname

    if not hostname:
      return ""

    # 移除 www.
    hostname = hostname.replace("www.", "")

    parts = hostname.split(".")

    if len(parts) < 2:
      return parts[0]

    if parts[1] in ["com", "tw", "net"]:
      return parts[0]

    return parts[0] + "-" + parts[1]

  except (IndexError, AttributeError):
    return ""


def parse_agoda_hotel_id(html):
  # """
  # 從 Agoda 網頁解析 hotel_id
  # """

  if not html:
    return ""

  matched = re.search(r"hotel_id=(\d+)", html)
  if matched:
    return matched.group(1)
  else:
    return ""


def extract_booking_slug(url):
  # ""解析 Booking slug"
  #
  # """

  if not url:
    return ""
  try:
    path = urlparse(url).path
    # /hotel/tw/grand-hyatt-taipei.zh-tw.html -> grand-hyatt-taipei
    return path.rstrip("/").split("/")[-1].split(".")[0]
  except (IndexError, AttributeError):
    return ""


def extract_kkday_slug(url):
  # """
  # 解析 KKday slug
  # """

  if not url:
    return ""
  try:
    path = urlparse(url).path
    if "product/" not in path:
        return ""
    # /zh-tw/product/28820 -> 28820
    return path.split("product/")[1].split("/")[0]
  except (IndexError, AttributeError):
    return ""


def extract_klook_slug(url):
  # """
  # 解析 Klook slug
  # """

  if not url:
    return ""
  try:
    path = urlparse(url).path
    if "detail/" not in path:
      return ""
    # https://www.klook.com/zh-TW/hotels/detail/569734-harbour-10-hotel/ -> 569734-harbour-10-hotel
    return path.split("detail/")[1].split("/")[0]
  except (IndexError, AttributeError):
    return ""


# ==========================================================
# API
# 外部服務
# ==========================================================

class GoogleAPI:
  def __init__(self, settings, session):
    self.key = settings.PLACES_KEY
    self.cx = settings.CX
    self.session = session


  async def search_place_id(self, query):
    # """
    # 透過 Google Places 找 place_id
    # """

    url = "https://places.googleapis.com/v1/places:searchText"
    headers = {
      "Content-Type": "application/json",
      "X-Goog-Api-Key": self.key,
      "X-Goog-FieldMask": "places.id",
      "Accept-Language": "zh-TW"
    }

    try:
      async with self.session.post(
        url,
        headers=headers,
        json={"textQuery": query},
        timeout=aiohttp.ClientTimeout(total=10)
      ) as response:
        response.raise_for_status()
        data = await response.json()

    except (aiohttp.ClientError, asyncio.TimeoutError, ValueError):
        return None

    places = data.get("places", [])
    if places and "id" in places[0]:
      return places[0]["id"]
    return None


  async def get_details(self, place_id):
    # """
    # 取得 Google Places 詳細資料
    # """

    url = f"https://places.googleapis.com/v1/places/{place_id}"
    headers = {
      "X-Goog-Api-Key": self.key,
      "X-Goog-FieldMask":
        "formattedAddress,internationalPhoneNumber,"
        "websiteUri,rating,userRatingCount,location",
      "Accept-Language": "zh-TW"
    }

    try:
      async with self.session.get(
        url,
        headers=headers,
        timeout=aiohttp.ClientTimeout(total=10)
      ) as response:
        response.raise_for_status()
        data = await response.json()

    except (aiohttp.ClientError, asyncio.TimeoutError, ValueError):
        return None

    return data


  async def search(self, query):
    # """
    # Google CSE 搜尋
    # """

    url = "https://www.googleapis.com/customsearch/v1"
    params = {
      "key": self.key,
      "cx": self.cx,
      "q": query,
      "num": 10
    }

    try:
      async with self.session.get(
        url,
        params=params,
        timeout=aiohttp.ClientTimeout(total=10)
      ) as response:
        response.raise_for_status()
        data = await response.json()

    except (aiohttp.ClientError, asyncio.TimeoutError, ValueError):
        return []

    return data.get("items", [])


class AgodaAPI:
  def __init__(self, settings, session):
    self.api_key = settings.AGODA_API_KEY
    self.checkin = settings.CHECKIN
    self.checkout = settings.CHECKOUT
    self.markup = settings.PRICE_MARKUP
    self.session = session


  async def get_hotel_id_from_url(self, url):
    # """
    # 抓取 Agoda 頁面並解析 hotel_id
    # """

    if not url:
      return None

    try:
      async with self.session.get(
        url,
        timeout=aiohttp.ClientTimeout(total=10)
      ) as response:
        response.raise_for_status()
        html = await response.text()

      # 呼叫 Utils 解析
      hotel_id = parse_agoda_hotel_id(html)

      return hotel_id

    except (aiohttp.ClientError, asyncio.TimeoutError):
      return None


  async def get_rate(self, hotel_id, adults):
    # """
    # 查詢指定人數房價
    # """

    if not hotel_id:
      return None

    url = "http://affiliateapi7643.agoda.com/affiliateservice/lt_v1"

    payload = {
      "criteria": {
        "hotelId": [int(hotel_id)],
        "checkInDate": self.checkin,
        "checkOutDate": self.checkout,
        "additional": {
          "currency": "TWD",
          "language": "zh-tw",
          "occupancy": {
            "numberOfAdult": adults,
            "numberOfChildren": 0
          }
        }
      }
    }

    headers = {
      "Authorization": f"{self.api_key}",
      "Accept-Encoding": "gzip,deflate",
      "Content-Type": "application/json"
    }



    try:
      async with self.session.post(
        url,
        json=payload,
        headers=headers,
        timeout=aiohttp.ClientTimeout(total=10)
      ) as response:
        response.raise_for_status()
        data = await response.json()

    except (aiohttp.ClientError, asyncio.TimeoutError, ValueError):
        return None

    results = data.get("results")
    if not results:
      return None

    daily_rate = results[0].get("dailyRate")
    if daily_rate is None:
      return None

    try:
      base_price = float(daily_rate)
    except (TypeError, ValueError):
      return None

    return int(base_price * self.markup)


# ==========================================================
# DATA MODEL
# 資料模型
# ==========================================================

@dataclass
class Hotel:
  name: str
  address: Optional[str] = None
  phone: Optional[str] = None
  website: Optional[str] = None
  slug: Optional[str] = None

  coordinates_lat: Optional[float] = None
  coordinates_lng: Optional[float] = None
  google_rating: Optional[float] = None
  google_reviews_count: Optional[int] = None

  # 預留欄位
  place_slug: Optional[str] = None
  labels: Optional[str] = None
  order: Optional[int] = None
  show_on_homepage: bool = False

  price_double_room: Optional[int] = None
  price_quad_room: Optional[int] = None

  agoda_slug: Optional[str] = None
  booking_slug: Optional[str] = None
  klook_slug: Optional[str] = None
  kkday_slug: Optional[str] = None

  # 轉成 dictionary，方便輸出 CSV
  def to_dict(self):
    return asdict(self)


# ==========================================================
# SERVICE
# 系統邏輯
# ==========================================================

class HotelService:
  def __init__(self, settings, session, semaphore):
    self.settings = settings
    self.google = GoogleAPI(settings, session)
    self.agoda = AgodaAPI(settings, session)
    self.semaphore = semaphore


  async def build(self, name):
    async with self.semaphore:
      # 避免同時間大量 API 呼叫導致限流
      await asyncio.sleep(random.uniform(0.2, 0.6))

      hotel = Hotel(name=name)

      await self._load_google_profile(hotel)
      await self._resolve_platform_slugs(hotel)
      await self._load_room_prices(hotel)

      return hotel


  async def _load_google_profile(self, hotel):
    # """
    # Google 基本資料
    # """

    try:
      place_id = await self.google.search_place_id(hotel.name)
      if not place_id:
        return

      detail = await self.google.get_details(place_id) or {}
    except Exception as e:
      print(f"[ERROR] Google profile load failed - {hotel.name}: {e}")
      return

    hotel.address = normalize_address(detail.get("formattedAddress"))
    hotel.phone = normalize_phone(detail.get("internationalPhoneNumber"))
    hotel.website = detail.get("websiteUri")
    hotel.slug = generate_slug(hotel.website) if hotel.website else ""
    hotel.google_rating = detail.get("rating")
    hotel.google_reviews_count = detail.get("userRatingCount")

    loc = detail.get("location", {})
    hotel.coordinates_lat = loc.get("latitude")
    hotel.coordinates_lng = loc.get("longitude")


  async def _resolve_platform_slugs(self, hotel):
    # """
    # 解析各平台的 slug
    # """

    # 透過 Google 搜尋結果解析各訂房平台連結
    try:
      items = await self.google.search(f"{hotel.name} 住宿")
    except Exception as e:
      print(f"[ERROR] Platform links search failed - {hotel.name}: {e}")
      return

    agoda_url = booking_url = klook_url = kkday_url = ""

    # 根據網址特徵判斷平台
    for item in items:
      link = item.get("link", "")
      title = item.get("title", "")
      snippet = item.get("snippet", "")

      if "agoda.com" in link and "/hotel/" in link:
        agoda_url = link

      if "booking.com/hotel/" in link:
        booking_url = link

      if "klook.com" in link and "hotels/detail/" in link:
        klook_url = link

      if "kkday.com" in link and "product" in link:
        # 過濾不符合條件的 KKday 連結
        if self.settings.KKDAY_BAD_KEYWORD in title or \
          self.settings.KKDAY_BAD_KEYWORD in snippet:
          continue
        kkday_url = link

    # 解析各平台 slug
    hotel.agoda_slug = await self.agoda.get_hotel_id_from_url(agoda_url)
    hotel.booking_slug = extract_booking_slug(booking_url)
    hotel.klook_slug = extract_klook_slug(klook_url)
    hotel.kkday_slug = extract_kkday_slug(kkday_url)


  async def _load_room_prices(self, hotel):
    # """
    # Agoda 房價
    # """
    if not hotel.agoda_slug:
      return

    try:
      hotel.price_double_room = await self.agoda.get_rate(hotel.agoda_slug, 2)
      hotel.price_quad_room = await self.agoda.get_rate(hotel.agoda_slug, 4)
    except Exception as e:
      print(f"[ERROR] Agoda price load failed - {hotel.name}: {e}")


# ==========================================================
# PIPELINE
# 流程控制
# ==========================================================

async def run():
  settings = Settings()

  semaphore = asyncio.Semaphore(settings.MAX_CONCURRENCY)

  # 建立共用 ClientSession 以重用連線池提升效能
  async with aiohttp.ClientSession() as session:
    # 建立 Service
    service = HotelService(settings, session, semaphore)

    # 建立所有非同步任務
    tasks = []
    for name in settings.HOTEL_LIST:
      task = service.build(name)
      tasks.append(task)

    # 同時執行所有任務
    results = await asyncio.gather(*tasks, return_exceptions=True)

    # 收集結果
    hotels = []
    for result in results:
      if isinstance(result, Exception):
        print(f"[ERROR] {result}")
      else:
        hotels.append(result)
        print(f"完成：{result.name}")

  # 整理成 DataFrame
  df = pd.DataFrame(hotel.to_dict() for hotel in hotels)

  # 指定 CSV 欄位順序
  columns = [
    "name",
    "address",
    "phone",
    "website",
    "slug",
    "coordinates_lat",
    "coordinates_lng",
    "google_rating",
    "google_reviews_count",

    "place_slug",
    "labels",
    "order",
    "show_on_homepage",

    "price_double_room",
    "price_quad_room",
    "agoda_slug",
    "booking_slug",
    "klook_slug",
    "kkday_slug",
  ]

  # 固定欄位順序，缺少欄位自動補 NaN
  df = df.reindex(columns=columns)

  # 排序 (將 NaN 轉為空字串前先排序以避免型別問題)
  df = df.sort_values(
    by=["google_rating", "google_reviews_count"],
    ascending=[False, False],
    na_position="last"
  )

  # 將 NaN 轉為空字串
  df = df.fillna("")

  # 輸出 CSV
  df.to_csv(settings.CSV_FILENAME, index=False, encoding="utf-8-sig")

  files.download(settings.CSV_FILENAME)


# ==========================================================
# EXECUTE
# 執行
# ==========================================================

await run()

print("全部處理完成")

完成：北投亞太飯店
完成：北投馥悅溫泉酒店
完成：曲水會館
完成：北投老爺酒店
完成：北投晶泉丰旅
完成：北投麗禧溫泉酒店
完成：大地酒店
完成：北投日勝生加賀屋
完成：北投春天酒店
完成：北投天玥泉溫泉會館


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

全部處理完成
